In [ ]:
# --- Section 1: Import Libraries and Configure Environment ---
from dataclasses import dataclass
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch

from training.common import set_seed


@dataclass
class Config:
    DATA_ROOT: str = "/kaggle/input/animals10/raw-img"
    CKPT_DIR: str = "/kaggle/working/checkpoints"
    RESULT_DIR: str = "/kaggle/working/results"
    BATCH_SIZE: int = 32
    EPOCHS_MAE: int = 50
    EPOCHS_UNET: int = 50
    EPOCHS_CLS: int = 20
    LR: float = 1e-4
    MASK_RATIO: float = 0.75
    IMG_SIZE: int = 224
    SEED: int = 42
    NUM_WORKERS: int = 2


cfg = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(cfg.SEED)

Path(cfg.CKPT_DIR).mkdir(parents=True, exist_ok=True)
Path(cfg.RESULT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Device: {device}")
print(f"Dataset root: {cfg.DATA_ROOT}")
print(f"Checkpoint dir: {cfg.CKPT_DIR}")
print(f"Result dir: {cfg.RESULT_DIR}")

In [ ]:
# --- Section 2: Load and Inspect the Dataset ---
from collections import Counter
from PIL import Image

from data.animals10 import build_split, discover_class_directories

class_dirs = discover_class_directories(cfg.DATA_ROOT)
split = build_split(cfg.DATA_ROOT, val_fraction=0.2, seed=cfg.SEED)
idx_to_class = {index: class_name for class_name, index in split.class_to_idx.items()}

print(f"Classes found: {len(class_dirs)}")
print("Class names:", list(split.class_to_idx.keys()))
print(f"Train samples: {len(split.train_samples)}")
print(f"Val samples: {len(split.val_samples)}")

train_counts = Counter(label for _, label in split.train_samples)
val_counts = Counter(label for _, label in split.val_samples)
print("Train distribution:", {idx_to_class[idx]: count for idx, count in train_counts.items()})
print("Val distribution:", {idx_to_class[idx]: count for idx, count in val_counts.items()})

sample_paths = [path for path, _ in split.train_samples[:4]]
fig, axes = plt.subplots(1, len(sample_paths), figsize=(16, 4))
for axis, sample_path in zip(axes, sample_paths):
    with Image.open(sample_path) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(Path(sample_path).parent.name)
    axis.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# --- Section 3: Define Image Preprocessing and Augmentation ---
from torchvision import transforms as T
from data.animals10 import IMAGENET_MEAN, IMAGENET_STD

train_transform = T.Compose(
    [
        T.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=10),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

val_transform = T.Compose(
    [
        T.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

print(train_transform)
print(val_transform)

In [ ]:
# --- Section 4: Build Data Loaders ---
from data.animals10 import build_dataloaders

train_loader, val_loader, split = build_dataloaders(
    root=cfg.DATA_ROOT,
    batch_size=cfg.BATCH_SIZE,
    image_size=cfg.IMG_SIZE,
    val_fraction=0.2,
    seed=cfg.SEED,
    num_workers=cfg.NUM_WORKERS,
    train_transform=train_transform,
    val_transform=val_transform,
)

idx_to_class = {index: class_name for class_name, index in split.class_to_idx.items()}
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
first_batch = next(iter(train_loader))
print(f"First batch image shape: {tuple(first_batch[0].shape)}")
print(f"First batch label shape: {tuple(first_batch[1].shape)}")

In [ ]:
# --- Section 5: Implement the MAE Encoder-Decoder Model ---
from mae_core import load_mae_model, load_mae_processor, reconstruct_image
from training.mae_trainer import reconstruct_mae_images

mae_processor = load_mae_processor()
mae_model = load_mae_model(mask_ratio=cfg.MASK_RATIO).to(device)
mae_model.eval()

sample_images = first_batch[0][:2].to(device)
with torch.no_grad():
    mae_outputs = mae_model(pixel_values=sample_images)
    mae_loss = mae_outputs.loss.item()

print(f"MAE loss on first batch: {mae_loss:.6f}")
print("MAE model ready for continual pre-training and reconstruction.")

In [ ]:
# --- Section 6: Add the Classification Head ---
from transformers import ViTForImageClassification
from training.classification import load_mae_encoder_weights_into_classifier

classification_model = ViTForImageClassification.from_pretrained(
    "facebook/vit-mae-base",
    num_labels=len(split.class_to_idx),
    ignore_mismatched_sizes=True,
).to(device)

classification_model = load_mae_encoder_weights_into_classifier(
    classification_model,
    mae_checkpoint_path=None,
)

print("Classification head attached.")
print(classification_model.__class__.__name__)

In [ ]:
# --- Section 7: Define Loss Functions and Optimizer ---
import torch.nn.functional as F

from models.unet import UNet
from training.common import create_grad_scaler
from training.unet import apply_patch_mask

unet_model = UNet().to(device)
mae_optimizer = torch.optim.AdamW(mae_model.parameters(), lr=cfg.LR)
unet_optimizer = torch.optim.AdamW(unet_model.parameters(), lr=cfg.LR)
cls_optimizer = torch.optim.AdamW(classification_model.parameters(), lr=cfg.LR)
mae_scaler = create_grad_scaler(device)
unet_scaler = create_grad_scaler(device)
cls_scaler = create_grad_scaler(device)

reconstruction_loss = torch.nn.MSELoss()
classification_loss = torch.nn.CrossEntropyLoss()

print("Optimizers and loss functions are ready.")

In [ ]:
# --- Section 8: Train the Model ---
from training.mae_trainer import MAETrainer
from training.unet import UNetReconstructionTrainer
from training.classification import ViTClassifierTrainer

# Smoke-training hooks. Increase epochs on Kaggle once the data path is validated.
mae_trainer = MAETrainer(mae_model, mae_optimizer, device, mask_ratio=cfg.MASK_RATIO, scaler=mae_scaler)
mae_train_loss = mae_trainer.train_epoch(train_loader)
mae_val_loss = mae_trainer.evaluate_epoch(val_loader)
print(f"MAE train loss: {mae_train_loss:.6f} | MAE val loss: {mae_val_loss:.6f}")

unet_trainer = UNetReconstructionTrainer(unet_model, unet_optimizer, device, mask_ratio=cfg.MASK_RATIO, scaler=unet_scaler)
unet_train_loss = unet_trainer.train_epoch(train_loader)
unet_val_loss = unet_trainer.evaluate_epoch(val_loader)
print(f"U-Net train loss: {unet_train_loss:.6f} | U-Net val loss: {unet_val_loss:.6f}")

cls_trainer = ViTClassifierTrainer(classification_model, cls_optimizer, device, scaler=cls_scaler)
cls_train_loss, cls_train_acc = cls_trainer.train_epoch(train_loader)
cls_val_loss, cls_val_acc = cls_trainer.evaluate_epoch(val_loader)
print(f"CLS train loss: {cls_train_loss:.6f} | train acc: {cls_train_acc:.4f}")
print(f"CLS val loss: {cls_val_loss:.6f} | val acc: {cls_val_acc:.4f}")

In [ ]:
# --- Section 9: Evaluate Reconstruction and Classification Metrics ---
from training.evaluation import compare_reconstruction_on_batch, evaluate_classifier

comparison_batch = next(iter(val_loader))
comparison_image_path = Path(cfg.RESULT_DIR) / "comparison" / "sample_comparison.png"
reconstruction_metrics = compare_reconstruction_on_batch(
    mae_model=mae_model,
    unet_model=unet_model,
    batch=comparison_batch,
    device=device,
    mask_ratio=cfg.MASK_RATIO,
    output_path=comparison_image_path,
)
classification_metrics = evaluate_classifier(classification_model, val_loader, device)

print("Reconstruction metrics:", reconstruction_metrics)
print("Classification metrics:", classification_metrics)

In [ ]:
# --- Section 10: Save Model Checkpoints and Inference Outputs ---
from training.common import save_json

summary_payload = {
    "device": str(device),
    "data_root": cfg.DATA_ROOT,
    "mae_val_mse": float(mae_val_loss),
    "unet_val_mse": float(unet_val_loss),
    "classification_val_loss": float(cls_val_loss),
    "classification_val_accuracy": float(cls_val_acc),
    "comparison_image_path": str(comparison_image_path),
}

save_json(Path(cfg.RESULT_DIR) / "run_summary.json", summary_payload)
print("Saved summary to:", Path(cfg.RESULT_DIR) / "run_summary.json")
print("Comparison image saved to:", comparison_image_path)
print("Checkpoint directories:", cfg.CKPT_DIR)